# 🧠 Brain Tumor Detection & Segmentation with Deep Learning

## TASK #1: Understand the Problem Statement and Business Case

**Goal:** Use AI to automatically detect and localize brain tumors in MRI scans.

Brain tumors are among the most deadly cancers. Early and accurate detection is critical for patient outcomes.
This project builds two deep-learning models:
1. **Classifier** – Detects *whether* a tumor is present (binary classification using ResNet50)
2. **Segmentation model** – Localizes *where* the tumor is (ResUNet pixel-level segmentation)

**Dataset:** [LGG MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)  
Contains brain MR images together with manual FLAIR abnormality segmentation masks from 110 patients.

**Environment:** Kaggle Notebook · GPU T4 x2 (multi-GPU via `tf.distribute.MirroredStrategy`)

## TASK #2: Import Libraries and Dataset

In [1]:
import os, glob, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import cv2
from skimage import io

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation, MaxPool2D,
    AveragePooling2D, Flatten, Dense, Dropout, Add,
    UpSampling2D, Concatenate
)
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

print('TensorFlow version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))
%matplotlib inline

2026-05-19 15:56:08.579964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779206168.811808      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779206168.881793      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779206169.400958      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779206169.400992      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779206169.400995      57 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


### Multi-GPU Setup
Kaggle T4 x2 uses `MirroredStrategy` to synchronise gradient updates across both GPUs.

In [2]:
# Multi-GPU strategy — automatically uses all available GPUs
strategy = tf.distribute.MirroredStrategy()
print(f'Number of devices in strategy: {strategy.num_replicas_in_sync}')

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices in strategy: 2


I0000 00:00:1779206218.988340      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779206218.994336      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


### Load the Dataset CSV

In [3]:
# ── Kaggle dataset path ──────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m'

# Build a dataframe from directory structure
# Each patient folder contains paired *_mask.tif and original .tif files
mask_files   = sorted(glob.glob(os.path.join(DATA_DIR, '**/*_mask.tif'), recursive=True))
image_files  = [f.replace('_mask', '') for f in mask_files]
patient_ids  = [os.path.basename(os.path.dirname(f)) for f in mask_files]

# has_mask = 1 if the mask is non-zero (tumour present)
has_mask = []
for m in mask_files:
    mask_arr = cv2.imread(m, cv2.IMREAD_GRAYSCALE)
    has_mask.append(1 if mask_arr is not None and mask_arr.max() > 0 else 0)

brain_df = pd.DataFrame({
    'patient_id': patient_ids,
    'image_path': image_files,
    'mask_path':  mask_files,
    'mask':       has_mask
})

print(brain_df.shape)
brain_df.head()

(3929, 4)


[ WARN:0@95.680] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m/TCGA_DU_A5TR_19970726/TCGA_DU_A5TR_19970726_19_mask.tif'): can't open/read file: check file path/integrity
[ WARN:0@95.680] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m/TCGA_DU_A5TR_19970726/TCGA_DU_A5TR_19970726_1_mask.tif'): can't open/read file: check file path/integrity
[ WARN:0@95.680] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m/TCGA_DU_A5TR_19970726/TCGA_DU_A5TR_19970726_20_mask.tif'): can't open/read file: check file path/integrity
[ WARN:0@95.681] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m/TCGA_DU_A5TR_19970726/TCGA_DU_A5TR_19970726_21_mask.tif'): can't open/read file: check file path/integrity
[ WARN:0@95.681] global loadsave.cpp:278 findDecoder imread_(

,patient_id,image_path,mask_path,mask
0,TCGA_CS_4941_19960909,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,0
1,TCGA_CS_4941_19960909,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,1
2,TCGA_CS_4941_19960909,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,1
3,TCGA_CS_4941_19960909,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,1
4,TCGA_CS_4941_19960909,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,/kaggle/input/datasets/mateuszbuda/lgg-mri-seg...,1


In [ ]:
brain_df.info()
print('\nClass distribution:')
print(brain_df['mask'].value_counts())

## TASK #3: Data Visualisation

In [ ]:
# Interactive bar chart with Plotly
counts = brain_df['mask'].value_counts()
fig = go.Figure([go.Bar(
    x=['No Tumor (0)', 'Tumor (1)'],
    y=counts.values,
    marker_color=['#2196F3', '#F44336'],
    opacity=0.8
)])
fig.update_layout(
    title='Tumour vs. Healthy MRI Distribution',
    xaxis_title='Class', yaxis_title='Count',
    template='plotly_dark'
)
fig.show()

In [ ]:
# Show random MRI + mask pairs
fig, axs = plt.subplots(6, 2, figsize=(14, 28))
sample_idx = random.sample(range(len(brain_df)), 6)
for row, i in enumerate(sample_idx):
    img  = cv2.cvtColor(cv2.imread(brain_df.image_path[i]), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(brain_df.mask_path[i], cv2.IMREAD_GRAYSCALE)
    axs[row][0].imshow(img);  axs[row][0].set_title('Brain MRI')
    axs[row][1].imshow(mask, cmap='gray')
    axs[row][1].set_title(f"Mask — {'Tumour' if brain_df.mask[i]==1 else 'Healthy'}")
    for ax in axs[row]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Visualise 12 SICK patients: MRI | Mask | MRI + Mask overlay
count = 0
fig, axs = plt.subplots(12, 3, figsize=(18, 48))
sick_df = brain_df[brain_df['mask'] == 1].reset_index(drop=True)
sample_sick = random.sample(range(len(sick_df)), 12)
for count, i in enumerate(sample_sick):
    img  = io.imread(sick_df.image_path[i])
    mask = io.imread(sick_df.mask_path[i])
    img_overlay = img.copy()
    img_overlay[mask == 255] = (255, 0, 0)

    axs[count][0].imshow(img);          axs[count][0].set_title('Brain MRI')
    axs[count][1].imshow(mask, cmap='gray'); axs[count][1].set_title('Mask')
    axs[count][2].imshow(img_overlay);  axs[count][2].set_title('MRI + Mask')
    for ax in axs[count]: ax.axis('off')
plt.tight_layout(); plt.show()

## TASK #4: CNN and ResNet — Theory

**Convolutional Neural Networks (CNNs)** learn hierarchical spatial features through shared filter banks.

**ResNets** (He et al., 2016) introduce *skip connections* that let gradients flow directly through identity
mappings, solving the vanishing-gradient problem and enabling networks 100+ layers deep.

```
Output = F(x) + x     ← residual block
```

ResNet-50 achieved **3.57 % top-5 error** on ImageNet (ensemble).

📄 Paper: https://arxiv.org/pdf/1512.03385.pdf

## TASK #5: Transfer Learning — Theory

Transfer learning reuses weights learned on a large dataset (ImageNet) for a new, smaller task.

**Strategy used here:**
1. Load ResNet50 pre-trained on ImageNet (frozen backbone)
2. Add a custom classification head
3. Train only the head → fast convergence with less data

**Key challenge:** *Negative transfer* — if the source and target domains are too different,
borrowed features may hurt performance.

## TASK #6: Train a Classifier to Detect Tumour Presence

### 6a. Prepare `tf.data` Pipeline (modern replacement for `ImageDataGenerator`)

In [ ]:
IMG_SIZE   = 256
BATCH_SIZE = 16 * strategy.num_replicas_in_sync  # scale batch with GPU count
AUTOTUNE   = tf.data.AUTOTUNE

# Prepare train / validation / test split
brain_df_train = brain_df.copy()
brain_df_train['mask'] = brain_df_train['mask'].astype(str)

train_df, test_df = train_test_split(brain_df_train, test_size=0.15, random_state=42)
train_df, val_df  = train_test_split(train_df,       test_size=0.15, random_state=42)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

In [ ]:
def load_and_preprocess(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.1)
    return img, label

def make_dataset(df, training=False):
    paths  = df['image_path'].values
    labels = tf.keras.utils.to_categorical(df['mask'].astype(int).values, num_classes=2)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(buffer_size=512)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)
print('Datasets ready.')

### 6b. Build the ResNet50 Classifier inside Multi-GPU Strategy

In [ ]:
with strategy.scope():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    )
    # Freeze backbone
    base_model.trainable = False

    # Classification head
    x = base_model.output
    x = AveragePooling2D(pool_size=(4, 4))(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        metrics=['accuracy']
    )

model.summary()

### 6c. Train the Classifier

In [ ]:
earlystop   = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
checkpoint  = ModelCheckpoint('classifier-resnet-weights.keras', monitor='val_loss',
                              save_best_only=True, verbose=1)
reduce_lr   = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    callbacks=[checkpoint, earlystop, reduce_lr]
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'],     label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Save model architecture
model_json = model.to_json()
with open('classifier-resnet-model.json', 'w') as f:
    f.write(model_json)
print('Classifier model saved.')

## TASK #7: Assess Classifier Performance

In [ ]:
# Predict on test set
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Ground truth (must match dataset ordering; test_ds is NOT shuffled)
y_true = test_df['mask'].astype(int).values[:len(y_pred)]

acc = accuracy_score(y_true, y_pred)
print(f'Test Accuracy: {acc:.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Tumor', 'Tumor'],
            yticklabels=['No Tumor', 'Tumor'])
plt.title('Confusion Matrix — Classifier')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
# Classification report
print(classification_report(y_true, y_pred, target_names=['No Tumor', 'Tumor']))

## TASK #8: ResUNet Architecture — Theory

**U-Net** is an encoder-decoder with skip connections, designed for biomedical image segmentation.

**ResUNet** replaces plain conv blocks with residual blocks, giving:
- Better gradient flow
- Fewer parameters for similar accuracy

Architecture summary:
```
Encoder (down-sampling):  residual blocks + MaxPool
Bottleneck:               deepest residual block
Decoder (up-sampling):    UpSampling + Concatenate + residual blocks
Output:                   1×1 Conv → Sigmoid (binary mask)
```

📄 Focal Tversky Loss paper: https://arxiv.org/abs/1810.07842

## TASK #9: Build the ResUNet Segmentation Model

### 9a. Custom Data Generator for Segmentation

In [ ]:
class SegDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_paths, mask_paths, batch_size=8, img_size=256, augment=False):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.batch_size  = batch_size
        self.img_size    = img_size
        self.augment     = augment
        self.indices     = np.arange(len(self.image_paths))

    def __len__(self):
        return max(1, len(self.image_paths) // self.batch_size)

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for i in batch_idx:
            img = cv2.cvtColor(cv2.imread(self.image_paths[i]), cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size)) / 255.0

            mask = cv2.imread(self.mask_paths[i], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.img_size, self.img_size))
            mask = (mask > 127).astype(np.float32)[..., np.newaxis]

            if self.augment and random.random() > 0.5:
                img  = np.fliplr(img)
                mask = np.fliplr(mask)
            if self.augment and random.random() > 0.5:
                img  = np.flipud(img)
                mask = np.flipud(mask)

            imgs.append(img)
            masks.append(mask)
        return np.array(imgs, dtype=np.float32), np.array(masks, dtype=np.float32)

In [ ]:
# Prepare segmentation split (tumour images only)
seg_df = brain_df[brain_df['mask'] == 1].reset_index(drop=True)
seg_train, seg_val_test = train_test_split(seg_df, test_size=0.20, random_state=42)
seg_val,   seg_test     = train_test_split(seg_val_test, test_size=0.50, random_state=42)

SEG_BATCH = 8 * strategy.num_replicas_in_sync

train_gen = SegDataGenerator(list(seg_train.image_path), list(seg_train.mask_path),
                              batch_size=SEG_BATCH, augment=True)
val_gen   = SegDataGenerator(list(seg_val.image_path),   list(seg_val.mask_path),
                              batch_size=SEG_BATCH)
print(f'Seg train: {len(seg_train)}  val: {len(seg_val)}  test: {len(seg_test)}')

### 9b. Focal Tversky Loss

In [ ]:
def tversky(y_true, y_pred, alpha=0.7, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    TP  = K.sum(y_true_f * y_pred_f)
    FP  = K.sum((1 - y_true_f) * y_pred_f)
    FN  = K.sum(y_true_f * (1 - y_pred_f))
    return (TP + smooth) / (TP + alpha * FP + (1 - alpha) * FN + smooth)

def tversky_loss(y_true, y_pred):
    return 1 - tversky(y_true, y_pred)

def focal_tversky(y_true, y_pred, gamma=0.75):
    tv = tversky(y_true, y_pred)
    return K.pow((1 - tv), gamma)

### 9c. ResUNet Architecture

In [ ]:
def resblock(X, filters):
    """Residual block: two conv layers with a shortcut connection."""
    X_skip = Conv2D(filters, (1, 1), kernel_initializer='he_normal')(X)
    X_skip = BatchNormalization()(X_skip)

    X = Conv2D(filters, (1, 1), kernel_initializer='he_normal')(X)
    X = BatchNormalization()(X)
    X = Activation('relu')(X)
    X = Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal')(X)
    X = BatchNormalization()(X)

    X = Add()([X, X_skip])
    X = Activation('relu')(X)
    return X

def upsample_concat(x, skip):
    x = UpSampling2D((2, 2))(x)
    return Concatenate()([x, skip])

def build_resunet(input_shape=(256, 256, 3)):
    X_input = Input(input_shape)

    # Encoder
    c1 = Conv2D(16, 3, activation='relu', padding='same', kernel_initializer='he_normal')(X_input)
    c1 = BatchNormalization()(c1)
    c1 = Conv2D(16, 3, activation='relu', padding='same', kernel_initializer='he_normal')(c1)
    c1 = BatchNormalization()(c1)
    p1 = MaxPool2D((2, 2))(c1)

    c2 = resblock(p1, 32);  p2 = MaxPool2D((2, 2))(c2)
    c3 = resblock(p2, 64);  p3 = MaxPool2D((2, 2))(c3)
    c4 = resblock(p3, 128); p4 = MaxPool2D((2, 2))(c4)

    # Bottleneck
    bn = resblock(p4, 256)

    # Decoder
    u1 = resblock(upsample_concat(bn,  c4), 128)
    u2 = resblock(upsample_concat(u1,  c3),  64)
    u3 = resblock(upsample_concat(u2,  c2),  32)
    u4 = resblock(upsample_concat(u3,  c1),  16)

    output = Conv2D(1, (1, 1), activation='sigmoid')(u4)
    return Model(inputs=X_input, outputs=output, name='ResUNet')

In [ ]:
with strategy.scope():
    model_seg = build_resunet()
    model_seg.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.05, epsilon=0.1),
        loss=focal_tversky,
        metrics=[tversky]
    )

model_seg.summary()

## TASK #10: Train the ResUNet Segmentation Model

In [ ]:
seg_checkpoint = ModelCheckpoint('ResUNet-weights.keras', monitor='val_loss',
                                  save_best_only=True, verbose=1)
seg_earlystop  = EarlyStopping(monitor='val_loss', patience=20,
                               restore_best_weights=True, verbose=1)
seg_reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

history_seg = model_seg.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[seg_checkpoint, seg_earlystop, seg_reduce_lr]
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_seg.history['tversky'],     label='Train Tversky')
ax1.plot(history_seg.history['val_tversky'], label='Val Tversky')
ax1.set_title('Tversky Index'); ax1.legend()
ax2.plot(history_seg.history['loss'],     label='Train Loss')
ax2.plot(history_seg.history['val_loss'], label='Val Loss')
ax2.set_title('Focal Tversky Loss'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Save segmentation model
model_seg_json = model_seg.to_json()
with open('ResUNet-model.json', 'w') as f:
    f.write(model_seg_json)
print('Segmentation model saved.')

## TASK #11: Assess Segmentation Model Performance

In [ ]:
def predict_and_visualize(df, classifier, segmenter, n=10, img_size=256):
    """Show: MRI | Original Mask | Predicted Mask | GT Overlay | Pred Overlay"""
    sick = df[df['mask'] == 1].reset_index(drop=True)
    samples = random.sample(range(len(sick)), min(n, len(sick)))
    fig, axs = plt.subplots(n, 5, figsize=(25, n * 5))
    titles = ['Brain MRI', 'Original Mask', 'AI Predicted Mask',
              'MRI + Ground Truth', 'MRI + AI Mask']
    for col, t in enumerate(titles):
        axs[0][col].set_title(t, fontsize=12, fontweight='bold')

    for row, i in enumerate(samples):
        img_bgr = cv2.imread(sick.image_path[i])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_r   = cv2.resize(img_rgb, (img_size, img_size))

        mask_orig = cv2.imread(sick.mask_path[i], cv2.IMREAD_GRAYSCALE)
        mask_r    = cv2.resize(mask_orig, (img_size, img_size))

        # Segmentation prediction
        inp         = (img_r / 255.0)[np.newaxis, ...]
        pred_mask   = segmenter.predict(inp, verbose=0)[0].squeeze()
        pred_binary = (pred_mask > 0.5).astype(np.uint8)

        # Overlays
        gt_overlay   = img_r.copy(); gt_overlay[mask_r == 255]   = (255,   0, 0)
        pred_overlay = img_r.copy(); pred_overlay[pred_binary==1] = (  0, 255, 0)

        for col, vis in enumerate([img_r, mask_r, pred_mask, gt_overlay, pred_overlay]):
            cmap = 'gray' if col in (1, 2) else None
            axs[row][col].imshow(vis, cmap=cmap)
            axs[row][col].axis('off')

    plt.tight_layout(); plt.show()

In [ ]:
predict_and_visualize(seg_test, model, model_seg, n=10)

In [ ]:
# Quantitative Dice / IoU on test set
def dice_score(y_true, y_pred, threshold=0.5):
    pred = (y_pred > threshold).astype(np.float32)
    inter = np.sum(y_true * pred)
    return 2 * inter / (np.sum(y_true) + np.sum(pred) + 1e-6)

dice_scores = []
for _, row in seg_test.iterrows():
    img   = cv2.resize(cv2.cvtColor(cv2.imread(row.image_path), cv2.COLOR_BGR2RGB), (256, 256)) / 255.0
    mask  = (cv2.resize(cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE), (256, 256)) > 127).astype(np.float32)
    pred  = model_seg.predict(img[np.newaxis], verbose=0)[0].squeeze()
    dice_scores.append(dice_score(mask, pred))

print(f'Mean Dice Score on test set: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}')

---

# 🎉 Excellent work!
You have successfully:
- Built a **ResNet50-based tumour classifier** with transfer learning
- Built a custom **ResUNet segmentation model** with Focal Tversky loss
- Leveraged **multi-GPU training** on Kaggle T4 x2 via `MirroredStrategy`
- Evaluated both models with accuracy, Dice score, and visual overlays

**Next steps:**
- Fine-tune the ResNet50 backbone (unfreeze top layers after head training)
- Try EfficientNetV2 or ConvNeXt as alternative backbones
- Add test-time augmentation (TTA) for better segmentation accuracy